# 01. Entendimiento del Dato

## Qué quiero lograr

En este cuaderno quiero entender el dataset antes de diseñar producto. Mi intención no es solo inspeccionar archivos: quiero validar qué representa realmente la señal, qué problemas de calidad existen y qué alcance de negocio puedo defender sin exagerar.

## Cómo voy a razonar

Voy a ir en este orden:

1. preparo un entorno reproducible,
2. cargo el raw en una vista por ventana y otra por punto,
3. describo la estructura real del export,
4. audito duplicados, solapes e incompletitud,
5. verifico si puedo construir una serie canónica,
6. traduzco el hallazgo a decisiones para dashboard y chatbot.

Quiero que el notebook muestre criterio, no solo outputs.


## Preparación del entorno

Antes de tocar los datos dejo el entorno listo para que el cuaderno funcione tanto si lo corro desde la raíz del repo como si lo corro desde la carpeta `notebooks/`.

También importo exactamente las funciones del pipeline que quiero poner a prueba. Hago esto porque no quiero duplicar lógica entre notebook y código productivo: prefiero que el análisis valide la misma implementación que luego consumirá el backend.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent

os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib"))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from scripts.process_availability_data import (
    RAW_DIR,
    STANDARD_CADENCE_SECONDS,
    STANDARD_WINDOW_DURATION_SECONDS,
    STANDARD_WINDOW_POINTS,
    annotate_windows,
    build_canonical_series,
    load_raw_windows_and_points,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 120)


## Helpers de lectura

Antes de entrar al análisis defino un par de helpers pequeños. Los uso para que las tablas salgan más legibles y para que las etiquetas de confianza o porcentaje no queden repetidas en varias celdas.

No quiero helpers abstractos; quiero funciones simples que ayuden a leer mejor el dato.


In [ ]:
def pct(value: float | int | None) -> str:
    # Formatea proporciones como porcentaje legible.
    if value is None or pd.isna(value):
        return "n/a"
    return f"{value * 100:.2f}%"


def confidence_flag(ratio: float | int | None) -> str:
    # Resume cobertura/calidad en una etiqueta corta.
    if ratio is None or pd.isna(ratio):
        return "sin evidencia"
    if ratio >= 0.95:
        return "alta"
    if ratio >= 0.80:
        return "media"
    return 'baja'


## 1. Cargo el raw en dos niveles

Aquí quiero probar algo clave: que puedo separar el raw en dos vistas complementarias sin perder trazabilidad.

- `windows_df` me sirve para entender cada archivo como una ventana exportada.
- `raw_points_df` me sirve para reconstruir la serie punto a punto.
- `canonical_df` me da una primera señal de si el dataset puede consolidarse sin contradicciones.

Si esta celda sale mal, todo el proyecto queda mal planteado.


In [ ]:
# Leo el raw y construyo la vista mínima que necesito para razonar con evidencia.
raw_windows_df, raw_points_df = load_raw_windows_and_points(RAW_DIR)
windows_df = annotate_windows(raw_windows_df)
canonical_df = build_canonical_series(windows_df, raw_points_df)

raw_load_summary = pd.DataFrame(
    [
        {
            "raw_files": len(windows_df),
            "raw_timestamp_cells": len(raw_points_df),
            "canonical_timestamps": len(canonical_df),
            "observed_start": canonical_df["timestamp"].min(),
            "observed_end": canonical_df["timestamp"].max(),
        }
    ]
)

display(raw_load_summary)


Con esta primera tabla quiero verificar que el volumen del raw sea coherente con la historia que voy a contar después. Me importa especialmente confirmar que el raw se pueda colapsar a una serie canónica sin destruir demasiada información.


## 2. Quiero entender la forma real del export

Antes de hablar de negocio necesito entender el contrato de facto del dataset. Con el siguiente bloque quiero probar:

- si todos los archivos traen la misma métrica,
- si la cadencia es estable,
- si la duración típica de ventana es consistente,
- si el archivo parece una exportación temporal y no una tabla por entidad.

Si el contrato es estable, después puedo construir analítica reutilizable con mucha más confianza.


In [ ]:
inventory_summary = pd.DataFrame(
    [
        {
            "unique_plot_names": windows_df["plot_name"].nunique(),
            "unique_metric_names": windows_df["metric_name"].nunique(),
            "metric_name_mode": windows_df["metric_name"].mode().iloc[0],
            "point_count_mode": int(windows_df["point_count"].mode().iloc[0]),
            "cadence_seconds_mode": int(windows_df["cadence_seconds"].mode().iloc[0]),
            "window_duration_mode": int(windows_df["window_duration_seconds"].mode().iloc[0]),
            "files_with_standard_shape": int(
                (
                    (windows_df["point_count"] == STANDARD_WINDOW_POINTS)
                    & (windows_df["cadence_seconds"] == STANDARD_CADENCE_SECONDS)
                    & (windows_df["window_duration_seconds"] == STANDARD_WINDOW_DURATION_SECONDS)
                ).sum()
            ),
        }
    ]
)

shape_profile = (
    windows_df[["point_count", "cadence_seconds", "window_duration_seconds"]]
    .describe()
    .round(2)
)

window_preview = windows_df[
    [
        "source_file",
        "metric_name",
        "window_start",
        "window_end",
        "point_count",
        "cadence_seconds",
        "window_duration_seconds",
        "first_value",
        "last_value",
    ]
].head(8)

display(inventory_summary)
display(shape_profile)
display(window_preview)


Mi lectura aquí tiene que ser concreta. Si veo una sola métrica, una cadencia muy dominante y ventanas con forma repetida, entonces empiezo a ver el dataset como una serie temporal agregada exportada por tramos. Eso cambia mucho el tipo de dashboard y el tipo de bot que vale la pena construir.


## 3. Quiero medir la calidad del raw antes de sacar conclusiones

Con este bloque voy a probar tres cosas:

1. cuántos duplicados exactos tengo,
2. cuántas ventanas están incompletas,
3. qué tan frecuente es cada problema en términos relativos.

Hago esto porque un insight sin contexto de calidad es una fuente potencial de mala interpretación.


In [ ]:
quality_snapshot = pd.DataFrame(
    [
        {
            "duplicate_window_groups": windows_df.loc[
                windows_df["is_duplicate_window"], "window_fingerprint"
            ].nunique(),
            "duplicate_window_records": int(windows_df["is_duplicate_window"].sum()),
            "duplicate_window_ratio": windows_df["is_duplicate_window"].mean(),
            "incomplete_window_records": int(windows_df["is_incomplete_window"].sum()),
            "incomplete_window_ratio": windows_df["is_incomplete_window"].mean(),
            "canonical_windows": int(windows_df["is_canonical_window"].sum()),
        }
    ]
).assign(
    duplicate_window_ratio=lambda frame: frame["duplicate_window_ratio"].map(pct),
    incomplete_window_ratio=lambda frame: frame["incomplete_window_ratio"].map(pct),
)

duplicate_windows = windows_df.loc[
    windows_df["is_duplicate_window"],
    [
        "source_file",
        "window_start",
        "window_end",
        "point_count",
        "duplicate_group_size",
        "duplicate_rank",
    ],
].head(12)

incomplete_windows = windows_df.loc[
    windows_df["is_incomplete_window"],
    [
        "source_file",
        "window_start",
        "window_end",
        "point_count",
        "cadence_seconds",
        "window_duration_seconds",
    ],
].head(12)

display(quality_snapshot)
display(duplicate_windows)
display(incomplete_windows)


Aquí quiero forzarme a pensar con más matiz. Un duplicado exacto suele ser un problema administrativo tratable. Una ventana incompleta ya es más delicada: puede sesgar cobertura, medias, anomalías y comparaciones si no la hago visible en el producto.


## 4. Quiero bajar la calidad a nivel de día

No me basta con saber que existen problemas. Quiero ver cómo se distribuyen en el tiempo, porque eso me ayuda a decidir si el riesgo es estructural o puntual.

En particular, quiero probar si hay días con peor cobertura o con más ventanas incompletas, porque esos días después requerirán etiquetas de confianza en dashboard y bot.


In [ ]:
window_day_profile = (
    windows_df.groupby("window_day")
    .agg(
        file_count=("source_file", "size"),
        unique_windows=("window_fingerprint", "nunique"),
        duplicate_records=("is_duplicate_window", "sum"),
        incomplete_records=("is_incomplete_window", "sum"),
        observed_points=("point_count", "sum"),
    )
    .reset_index()
)
window_day_profile["duplicate_ratio"] = (
    window_day_profile["duplicate_records"] / window_day_profile["file_count"]
)
window_day_profile["incomplete_ratio"] = (
    window_day_profile["incomplete_records"] / window_day_profile["file_count"]
)
window_day_profile["expected_points_if_all_windows_standard"] = (
    window_day_profile["file_count"] * STANDARD_WINDOW_POINTS
)
window_day_profile["observed_points_ratio"] = (
    window_day_profile["observed_points"]
    / window_day_profile["expected_points_if_all_windows_standard"]
)
window_day_profile["coverage_confidence"] = window_day_profile[
    "observed_points_ratio"
].map(confidence_flag)

display(window_day_profile.round(4))

fig, ax = plt.subplots(figsize=(12, 4))
window_day_profile.plot(
    x="window_day",
    y=["duplicate_ratio", "incomplete_ratio", "observed_points_ratio"],
    kind="bar",
    ax=ax,
    title="Calidad diaria del raw exportado",
)
ax.set_xlabel("Día")
ax.set_ylabel("Ratio")
plt.xticks(rotation=45)
plt.tight_layout()


Esta celda ya conecta análisis con producto. Si la calidad cambia por día, entonces una UI honesta debería mostrar cobertura, confianza o advertencias por período consultado. No quiero que el usuario crea que todos los días tienen la misma solidez analítica si no es cierto.


## 5. Quiero probar si la serie canónica es defendible

Esta es la prueba más importante del cuaderno. Voy a revisar si los timestamps que aparecen varias veces traen el mismo valor o no.

Lo que busco demostrar es muy simple: si los solapes no introducen conflictos de valor, entonces puedo consolidar una serie canónica sin inventar verdad. Si hubiera conflictos, tendría que cambiar de estrategia.


In [ ]:
canonical_checks = pd.DataFrame(
    [
        {
            "timestamps_unique": canonical_df["timestamp"].is_unique,
            "timestamps_sorted": canonical_df["timestamp"].is_monotonic_increasing,
            "overlapping_timestamps": int((canonical_df["timestamp_occurrences"] > 1).sum()),
            "conflicting_timestamps": int(canonical_df["has_conflicting_values"].sum()),
            "max_occurrences_for_same_timestamp": int(
                canonical_df["timestamp_occurrences"].max()
            ),
        }
    ]
)

timestamp_multiplicity = (
    canonical_df["timestamp_occurrences"]
    .value_counts()
    .sort_index()
    .rename_axis("occurrences")
    .reset_index(name="timestamps")
)

overlap_examples = canonical_df.loc[
    canonical_df["timestamp_occurrences"] > 1,
    [
        "timestamp",
        "value",
        "timestamp_occurrences",
        "unique_values",
        "has_conflicting_values",
    ],
].head(12)

assert canonical_df["timestamp"].is_unique, "La serie canónica no quedó única por timestamp."
assert canonical_df["timestamp"].is_monotonic_increasing, "La serie canónica no quedó ordenada."
assert not canonical_df["has_conflicting_values"].any(), "Existen conflictos de valor en timestamps solapados."

display(canonical_checks)
display(timestamp_multiplicity)
display(overlap_examples)


Aquí es donde realmente me tranquilizo. Si los solapes repiten el mismo valor, la deduplicación deja de ser una apuesta y pasa a ser una decisión técnica defendible. Eso me da una fuente de verdad operativa para el resto del proyecto.


## 6. Quiero dejar explícito qué hipótesis quedan soportadas

Más que cerrar con intuiciones, quiero terminar con un pequeño registro de hipótesis. Me sirve para separar lo que ya demostré, lo que solo interpreto de manera razonable y lo que todavía no debería prometer.


In [ ]:
hypothesis_register = pd.DataFrame(
    [
        {
            "hypothesis": "La unidad analítica correcta es una serie temporal agregada.",
            "status": "supported" if windows_df["metric_name"].nunique() == 1 else "needs_review",
            "why": "Todos los archivos observados comparten una sola métrica y vienen como ventanas temporales, no como entidades de negocio.",
        },
        {
            "hypothesis": "La cadencia dominante del dato es de 10 segundos.",
            "status": "supported" if int(windows_df["cadence_seconds"].mode().iloc[0]) == STANDARD_CADENCE_SECONDS else "needs_review",
            "why": "La moda de cadence_seconds coincide con el estándar esperado del export.",
        },
        {
            "hypothesis": "El raw contiene problemas de calidad que deben exponerse al producto.",
            "status": "supported" if windows_df["is_incomplete_window"].any() or windows_df["is_duplicate_window"].any() else "not_observed",
            "why": "Se detectan duplicados e incompletitud, por lo que el dashboard y el bot deben manejar confianza/cobertura.",
        },
        {
            "hypothesis": "La serie puede consolidarse sin conflictos de valor por timestamp.",
            "status": "supported" if not canonical_df["has_conflicting_values"].any() else "blocked",
            "why": "Los timestamps repetidos no introducen valores contradictorios en la serie canónica.",
        },
        {
            "hypothesis": "El dataset soporta análisis por tienda o ranking de entidades.",
            "status": "not_supported",
            "why": "No observo dimensiones de tienda, merchant o entidad individual en el raw analizado.",
        },
    ]
)

display(hypothesis_register)


## 7. Qué me llevo al siguiente notebook

Lo que me llevo de esta fase es bastante claro:

- sí puedo construir una verdad operativa por timestamp,
- no debo prometer granularidad por tienda,
- tengo que preservar metadata de calidad junto a la señal,
- me conviene construir agregados diarios y horarios para que el backend no consulte siempre la serie de 10 segundos,
- el dashboard y el bot deben poder hablar de cobertura y confianza, no solo de valores.

Con esta base, el siguiente notebook ya no será de exploración abierta: será de construcción deliberada de artefactos para producto.
